# HrudayAI — Phase 1: Speech Recognition (Google Colab Setup & Prototype)

## Objective

**Phase 1 (`phase1_voice`)** converts spoken rural patient health queries—across Indian languages and regional accents—into clean, structured text transcripts using **free, open-source speech recognition models** (with zero paid APIs).

This notebook serves as the starting point for Google Colab experimentation and verifies:
1. Python and Google Colab runtime environment.
2. Optional GPU availability (with safe CPU fallback).
3. Candidate open-source Indian-language ASR model selection (configuration only—**no large model downloads yet**).
4. Ephemeral sample audio upload and basic input validation.
5. A modular speech-to-text inference placeholder aligned with HrudayAI's Phase 1 → Phase 2 Python data contract.

> **Privacy Reminder:** Never commit private patient data, credentials, audio recordings, or model weights to GitHub.

## Expected Inputs & Outputs

To keep `phase1_voice`, `phase2_prediction`, and `phase3_voice_output` modular, Phase 1 exposes a consistent Python function interface.

### Expected Inputs
- **`audio_path`** (`str | pathlib.Path`): Path to an uploaded/local audio clip (supported formats: `.wav`, `.mp3`, `.flac`, `.ogg`, `.m4a`; recommended `16 kHz`, mono PCM `.wav` for ASR).
- **`language`** (`str`, optional): ISO language hint (e.g., `"hi"`, `"bn"`, `"ta"`, `"te"`, `"mr"`, `"kn"`, `"ml"`, `"gu"`, `"pa"`, `"en"`, or `"auto"`).
- **`model_key`** (`str`): Key identifying an open-source model configuration from `OPEN_SOURCE_ASR_CANDIDATES`.

### Expected Output (Phase 1 → Phase 2 Contract)
A Python `dict` structured for downstream symptom extraction and disease prediction:
```python
{
    "transcript": str,           # Recognized patient speech text
    "language": str,             # Detected or specified language code
    "confidence": float | None,  # Optional confidence score in [0.0, 1.0]
    "duration_seconds": float | None,  # Audio duration in seconds
    "model_id": str,             # Open-source Hugging Face / model identifier
    "device": str,               # "cuda" or "cpu"
    "status": str                # "ok" when inferred, "placeholder" before model wiring
}
```

## 1. Setup & Environment Verification

Verify Python runtime details, detect whether the notebook is running in Google Colab or locally, and inspect which ML/audio packages are pre-installed without installing or downloading heavy artifacts yet.

In [ ]:
import importlib.util
import os
import platform
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

print(f"Python version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")
print(f"Working dir    : {Path.cwd()}")
print(f"Running in     : {'Google Colab' if IN_COLAB else 'Local / Non-Colab Jupyter'}")

# Non-destructive check for common open-source ML/audio libraries
packages_to_check = ["torch", "torchaudio", "transformers", "librosa", "soundfile", "numpy"]
print("\nPackage availability check (no downloads performed):")
for pkg in packages_to_check:
    available = importlib.util.find_spec(pkg) is not None
    status = "installed" if available else "not installed"
    print(f"  - {pkg:<14}: {status}")

# Optional Colab note:
# When running in Colab with the private GitHub repo cloned, ensure the repo root
# is on sys.path so `phase1_voice.src` and `shared` can be imported directly:
#   REPO_ROOT = Path('/content/HrudayAI')
#   if REPO_ROOT.exists() and str(REPO_ROOT) not in sys.path:
#       sys.path.insert(0, str(REPO_ROOT))

## 2. GPU Availability Check

Check whether a CUDA GPU is available in the current runtime. HrudayAI does **not** assume a GPU is always present—if no GPU is detected, the notebook cleanly selects `cpu` execution.

In [ ]:
import importlib.util
import shutil
import subprocess

SELECTED_DEVICE = "cpu"

if importlib.util.find_spec("torch") is not None:
    import torch

    if torch.cuda.is_available():
        SELECTED_DEVICE = "cuda"
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"GPU detected   : {gpu_name} ({vram_gb:.2f} GB VRAM)")
    else:
        print("PyTorch installed, but no CUDA GPU detected. Using CPU fallback.")
else:
    print("PyTorch is not installed in this environment. Defaulting device to 'cpu'.")
    if shutil.which("nvidia-smi"):
        result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                capture_output=True, text=True, check=False)
        if result.returncode == 0 and result.stdout.strip():
            print(f"nvidia-smi GPU : {result.stdout.strip()}")
        else:
            print("nvidia-smi available, but no active GPU device reported.")

print(f"Selected device: {SELECTED_DEVICE}")

## 3. Open-Source Indian-Language Speech Recognition Model Selection

All candidate models below are **free and open-source** (no paid APIs). This section only defines and validates the model selection metadata—**it does not download model weights**.

Candidate families to evaluate once Phase 1 dependencies are finalized:
- **AI4Bharat IndicConformer / IndicWav2Vec**: Purpose-built open-source ASR models trained on Indian languages (e.g., Hindi, Bengali, Tamil, Telugu, Marathi, Gujarati, Kannada, Malayalam, Odia, Punjabi).
- **OpenAI Whisper (open-source weights, e.g. `small` / `medium`)**: Multilingual ASR with zero-shot support for major Indian languages and strong noise robustness.
- **Meta MMS (Massively Multilingual Speech)**: Open-source multilingual ASR supporting 1,100+ languages, including many low-resource regional languages.

In [ ]:
OPEN_SOURCE_ASR_CANDIDATES = {
    "indic_conformer_multilingual": {
        "model_id": "ai4bharat/indic-conformer-600m-multilingual",
        "family": "AI4Bharat IndicConformer",
        "license": "MIT / Open-Source (verify model card before production use)",
        "languages": ["hi", "bn", "ta", "te", "mr", "gu", "kn", "ml", "or", "pa", "as"],
        "expected_sample_rate_hz": 16000,
        "notes": "Strong Indian-language accuracy; best executed on Colab GPU."
    },
    "whisper_small": {
        "model_id": "openai/whisper-small",
        "family": "OpenAI Whisper (Open Weights)",
        "license": "MIT",
        "languages": ["hi", "bn", "ta", "te", "mr", "gu", "kn", "ml", "pa", "ur", "en"],
        "expected_sample_rate_hz": 16000,
        "notes": "Lightweight baseline; feasible on CPU for short clips and fast on T4 GPU."
    },
    "mms_1b_all": {
        "model_id": "facebook/mms-1b-all",
        "family": "Meta MMS",
        "license": "CC-BY-NC 4.0 (research/non-commercial; check license fit)",
        "languages": ["hin", "ben", "tam", "tel", "mar", "guj", "kan", "mal", "ory", "pan"],
        "expected_sample_rate_hz": 16000,
        "notes": "Broad regional language adapter coverage; requires GPU for larger runs."
    },
}


def select_asr_model(model_key: str) -> dict:
    """Return metadata for a selected open-source ASR candidate without downloading weights."""
    if model_key not in OPEN_SOURCE_ASR_CANDIDATES:
        valid_keys = ", ".join(OPEN_SOURCE_ASR_CANDIDATES.keys())
        raise ValueError(f"Unknown model_key '{model_key}'. Choose one of: {valid_keys}")
    return OPEN_SOURCE_ASR_CANDIDATES[model_key]


SELECTED_MODEL_KEY = "whisper_small"
selected_model_config = select_asr_model(SELECTED_MODEL_KEY)

print("Selected open-source ASR candidate (metadata only, no weights downloaded):")
for k, v in selected_model_config.items():
    print(f"  - {k}: {v}")

## 4. Sample Audio Upload & Validation

Upload a temporary sample audio clip in Google Colab (or point to a local non-committed test clip when running outside Colab). Keep all audio files out of Git.

In [ ]:
from pathlib import Path
import wave

SUPPORTED_AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}
SAMPLE_AUDIO_PATH = None
LANGUAGE_HINT = "hi"  # Change to target language code (e.g., 'hi', 'ta', 'bn', 'en', or 'auto')

# Toggle to True in Google Colab when ready to upload a sample clip interactively
ENABLE_COLAB_UPLOAD = False

if IN_COLAB and ENABLE_COLAB_UPLOAD:
    from google.colab import files  # type: ignore

    uploaded = files.upload()
    if uploaded:
        first_filename = next(iter(uploaded.keys()))
        SAMPLE_AUDIO_PATH = Path(first_filename).resolve()
else:
    # Optional local path for dry-run testing outside Colab (keep test audio outside Git tracking)
    local_candidate = Path("sample_audio.wav")
    if local_candidate.exists():
        SAMPLE_AUDIO_PATH = local_candidate.resolve()


def inspect_audio_input(audio_path: Path | None) -> dict:
    """Validate audio file extension and inspect basic WAV header properties if available."""
    if audio_path is None:
        return {
            "provided": False,
            "valid": False,
            "message": "No audio file uploaded yet. Set ENABLE_COLAB_UPLOAD=True in Colab or provide a local path."
        }

    if not audio_path.exists():
        return {"provided": True, "valid": False, "message": f"File not found: {audio_path}"}

    ext = audio_path.suffix.lower()
    if ext not in SUPPORTED_AUDIO_EXTENSIONS:
        return {
            "provided": True,
            "valid": False,
            "message": f"Unsupported format '{ext}'. Supported: {sorted(SUPPORTED_AUDIO_EXTENSIONS)}"
        }

    info = {
        "provided": True,
        "valid": True,
        "path": str(audio_path),
        "extension": ext,
        "size_bytes": audio_path.stat().st_size,
        "duration_seconds": None,
        "sample_rate_hz": None,
        "channels": None,
    }

    if ext == ".wav":
        with wave.open(str(audio_path), "rb") as wf:
            frames = wf.getnframes()
            rate = wf.getframerate()
            info["sample_rate_hz"] = rate
            info["channels"] = wf.getnchannels()
            info["duration_seconds"] = round(frames / float(rate), 3) if rate else None

    return info


audio_metadata = inspect_audio_input(SAMPLE_AUDIO_PATH)
print("Audio input status:", audio_metadata)

## 5. Speech-to-Text Inference Placeholder

Once Phase 1 dependencies and open-source model weights are finalized, the real inference logic will live in reusable Python modules under `phase1_voice/src/` and be imported here.

Below is the **placeholder inference function** returning the exact Phase 1 → Phase 2 output contract without downloading or executing a large model.

In [ ]:
def transcribe_audio_placeholder(
    audio_info: dict,
    model_config: dict,
    language: str = "auto",
    device: str = "cpu",
) -> dict:
    """Placeholder Phase 1 ASR inference function matching the Phase 1 -> Phase 2 data contract."""
    # TODO(phase1): Replace placeholder with open-source ASR pipeline from `phase1_voice.src`
    # after dependencies are selected in `phase1_voice/requirements.txt`.
    return {
        "transcript": "[PLACEHOLDER] Speech-to-text inference not yet wired.",
        "language": language,
        "confidence": None,
        "duration_seconds": audio_info.get("duration_seconds"),
        "model_id": model_config["model_id"],
        "device": device,
        "status": "placeholder",
    }


phase1_output = transcribe_audio_placeholder(
    audio_info=audio_metadata,
    model_config=selected_model_config,
    language=LANGUAGE_HINT,
    device=SELECTED_DEVICE,
)

print("Phase 1 output payload (ready for Phase 2 integration):")
for key, value in phase1_output.items():
    print(f"  - {key}: {value}")